<a href="https://colab.research.google.com/github/Haritha-reddie/Delhi-AQI-Intelligence-App/blob/main/delhi_aqi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌫️ Delhi AQI Intelligence App
### AI-powered Air Quality Analytics · Dashboard + Forecasting + Claude AI Chat


## ✅ CELL 1 — Install packages

In [ ]:
!pip install streamlit anthropic plotly prophet pyngrok localtunnel -q
print('✅ All packages installed!')

ERROR: Ignored the following versions that require a different python version: 0.55.2 Requires-Python <3.5
ERROR: Could not find a version that satisfies the requirement localtunnel (from versions: none)
ERROR: No matching distribution found for localtunnel
✅ All packages installed!


## ✅ CELL 2 — Create project folders

In [ ]:
import os

for folder in ['backend', 'ai', 'frontend', 'data']:
    os.makedirs(f'/content/{folder}', exist_ok=True)

open('/content/backend/__init__.py', 'w').close()
open('/content/ai/__init__.py', 'w').close()

print('✅ Folders created:')
print(os.listdir('/content'))

✅ Folders created:
['.config', 'data', 'ai', 'backend', 'frontend', 'sample_data']


## ✅ CELL 3 — Upload your CSV

In [ ]:
from google.colab import files
import shutil, os

print('👆 Click Choose Files and select your delhi_aqi CSV...')
uploaded = files.upload()

# Copy whichever file was uploaded to the standard path
for filename in uploaded.keys():
    shutil.copy(f'/content/{filename}', '/content/data/delhi_aqi.csv')
    print(f'✅ Copied "{filename}" → /content/data/delhi_aqi.csv')

print('Files in data/:', os.listdir('/content/data/'))

👆 Click Choose Files and select your delhi_aqi CSV...


Saving Delhi Air Quality Time Series Dataset(01-01-2025 to 15-05-2026).csv to Delhi Air Quality Time Series Dataset(01-01-2025 to 15-05-2026).csv
✅ Copied "Delhi Air Quality Time Series Dataset(01-01-2025 to 15-05-2026).csv" → /content/data/delhi_aqi.csv
Files in data/: ['delhi_aqi.csv']


## ✅ CELL 4 — Set your Anthropic API Key

In [ ]:
import os

# Paste your key from https://console.anthropic.com
os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-your-key-here'

key = os.environ.get('ANTHROPIC_API_KEY', '')
if key.startswith('sk-ant') and 'paste' not in key:
    print('✅ API key set!')
else:
    print('⚠️  Please replace with your real API key from console.anthropic.com')
    print('    AI Chat will be disabled until you do.')

✅ API key set!


## ✅ CELL 5 — Write backend/data_loader.py

In [ ]:
%%writefile /content/backend/data_loader.py
import pandas as pd
import numpy as np

DATA_PATH = '/content/data/delhi_aqi.csv'

def load_data():
    df = pd.read_csv(DATA_PATH)

    # Parse datetime
    df['datetime'] = pd.to_datetime(df['DateTime'], errors='coerce')
    df = df.dropna(subset=['datetime']).sort_values('datetime').reset_index(drop=True)

    # Clean values
    df['Values'] = pd.to_numeric(df['Values'], errors='coerce')
    df['Parameters'] = df['Parameters'].str.strip().str.lower()
    df['Locations'] = df['Locations'].str.strip()

    # ── Pivot: one row per (datetime, location), one column per parameter ──
    pivot = df.pivot_table(
        index=['datetime', 'Locations'],
        columns='Parameters',
        values='Values',
        aggfunc='mean'
    ).reset_index()

    pivot.columns.name = None
    pivot = pivot.rename(columns={'Locations': 'location'})

    # ── Create AQI column (use 'aqi' parameter if exists, else pm25) ──
    if 'aqi' in pivot.columns:
        pivot['aqi'] = pd.to_numeric(pivot['aqi'], errors='coerce')
    elif 'pm2.5' in pivot.columns:
        pivot['aqi'] = pivot['pm2.5']
    elif 'pm25' in pivot.columns:
        pivot['aqi'] = pivot['pm25']
    else:
        # Use first numeric column as proxy
        num_cols = pivot.select_dtypes(include=[np.number]).columns.tolist()
        if num_cols:
            pivot['aqi'] = pivot[num_cols[0]]

    print("✅ Data loaded! Shape:", pivot.shape)
    print("Columns:", pivot.columns.tolist())
    print("Locations:", pivot['location'].unique().tolist())
    return pivot

def filter_data(df, locations=None, start_date=None, end_date=None):
    if locations:    df = df[df['location'].isin(locations)]
    if start_date:   df = df[df['datetime'] >= pd.Timestamp(start_date)]
    if end_date:     df = df[df['datetime'] <= pd.Timestamp(end_date)]
    return df.reset_index(drop=True)

def get_locations(df):
    return sorted(df['location'].dropna().unique().tolist())

def get_pollutant_columns(df):
    exclude = {'datetime', 'location', 'aqi'}
    cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude]
    return ['aqi'] + cols if 'aqi' in df.columns else cols

def get_summary_stats(df, location=None):
    if location: df = df[df['location'] == location]
    stats = {}
    for col in df.select_dtypes(include=[np.number]).columns:
        s = df[col].dropna()
        if len(s):
            stats[col] = {
                'mean':   round(s.mean(), 2),
                'max':    round(s.max(), 2),
                'min':    round(s.min(), 2),
                'latest': round(s.iloc[-1], 2)
            }
    stats['_meta'] = {
        'total_records': len(df),
        'date_range': f"{df['datetime'].min().date()} to {df['datetime'].max().date()}",
        'locations': get_locations(df)
    }
    return stats

def get_aqi_category(v):
    if v <= 50:    return 'Good', '#00e400'
    elif v <= 100: return 'Satisfactory', '#ffff00'
    elif v <= 200: return 'Moderate', '#ff7e00'
    elif v <= 300: return 'Poor', '#ff0000'
    elif v <= 400: return 'Very Poor', '#8f3f97'
    else:          return 'Severe', '#7e0023'

Writing /content/backend/data_loader.py


## ✅ CELL 6 — Write backend/forecast.py

In [ ]:
%%writefile /content/backend/forecast.py
import pandas as pd
import numpy as np
import plotly.graph_objects as go

try:
    from prophet import Prophet
    PROPHET_OK = True
except ImportError:
    PROPHET_OK = False

def get_forecast(df, location=None, target_col='aqi', days=30):
    if location:
        df = df[df['location'] == location]

    series = df[['datetime', target_col]].rename(columns={'datetime':'ds', target_col:'y'})
    series = series.dropna().set_index('ds').resample('D').mean().reset_index().dropna()

    if len(series) < 10:
        return {'error': 'Not enough data (need at least 10 days).'}

    if PROPHET_OK:
        model = Prophet(yearly_seasonality=True, weekly_seasonality=True,
                        daily_seasonality=False, changepoint_prior_scale=0.05)
        model.fit(series)
        future   = model.make_future_dataframe(periods=days, freq='D')
        forecast = model.predict(future)
    else:
        last_val  = series['y'].rolling(min(14,len(series))).mean().iloc[-1]
        fut_dates = pd.date_range(series['ds'].iloc[-1] + pd.Timedelta(days=1), periods=days)
        forecast  = pd.DataFrame({'ds': pd.concat([series['ds'], pd.Series(fut_dates)]),
                                   'yhat': list(series['y']) + [last_val]*days,
                                   'yhat_upper': list(series['y']) + [last_val*1.1]*days,
                                   'yhat_lower': list(series['y']) + [last_val*0.9]*days})

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=series['ds'], y=series['y'],
                             name='Actual', line=dict(color='#4FC3F7', width=2)))
    fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat'],
                             name='Forecast', line=dict(color='#FF7043', width=2, dash='dash')))
    fig.add_trace(go.Scatter(
        x=pd.concat([forecast['ds'], forecast['ds'][::-1]]),
        y=pd.concat([forecast['yhat_upper'], forecast['yhat_lower'][::-1]]),
        fill='toself', fillcolor='rgba(255,112,67,0.15)',
        line=dict(color='rgba(0,0,0,0)'), name='Confidence Band'))
    fig.update_layout(title=f'{target_col.upper()} Forecast — Next {days} Days',
                      template='plotly_dark', hovermode='x unified',
                      margin=dict(l=40,r=20,t=60,b=40))

    future_only = forecast[forecast['ds'] > series['ds'].max()]
    return {'fig': fig,
            'avg_forecast': round(future_only['yhat'].mean(), 1),
            'max_forecast': round(future_only['yhat'].max(), 1),
            'min_forecast': round(future_only['yhat'].min(), 1),
            'days': days}

Writing /content/backend/forecast.py


## ✅ CELL 7 — Write ai/agent.py

In [ ]:
%%writefile /content/ai/agent.py
import os
import anthropic
import pandas as pd

SYSTEM_PROMPT = """You are an expert air quality analyst for Delhi, India.
You have access to real sensor data from Delhi monitoring stations (2025-2026).
Answer questions clearly using the data context provided.
Always explain AQI numbers in plain English and give health advice when relevant.
AQI Scale (India): 0-50 Good, 51-100 Satisfactory, 101-200 Moderate,
201-300 Poor, 301-400 Very Poor, 400+ Severe."""

def build_context(df, location=None):
    sub = df[df['location'] == location] if location and location != 'All Locations' else df
    lines = []
    for col in sub.select_dtypes(include='number').columns:
        s = sub[col].dropna()
        if len(s):
            lines.append(f'  {col.upper()}: mean={s.mean():.1f}, max={s.max():.1f}, min={s.min():.1f}, latest={s.iloc[-1]:.1f}')
    return f"""Dataset: {df['datetime'].min().date()} to {df['datetime'].max().date()}
Locations: {', '.join(str(l) for l in sub['location'].unique()[:8])}
Records: {len(sub):,}
Stats:
""" + '\n'.join(lines)

def ask_agent(question, df, location=None, chat_history=None):
    api_key = os.environ.get('ANTHROPIC_API_KEY', '')
    if not api_key or 'paste' in api_key:
        return '⚠️ Please set your ANTHROPIC_API_KEY.'
    try:
        client   = anthropic.Anthropic(api_key=api_key)
        messages = (chat_history or []) + [{'role': 'user', 'content': f'Data:\n{build_context(df, location)}\n\nQuestion: {question}'}]
        resp = client.messages.create(model='claude-sonnet-4-6', max_tokens=1024,
                                      system=SYSTEM_PROMPT, messages=messages)
        return resp.content[0].text
    except Exception as e:
        return f'⚠️ Error: {str(e)}'

def get_quick_insight(df, location=None):
    q = f"Give a 3-sentence summary of air quality {'in '+location if location else 'across Delhi'}. Include AQI level, health impact, and one recommendation."
    return ask_agent(q, df, location)

Writing /content/ai/agent.py


In [ ]:
!pip install anthropic prophet plotly -q
print('✅ Done!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.0/833.0 kB 17.6 MB/s eta 0:00:00
✅ Done!


## ✅ CELL 8 — Write frontend/app.py

In [ ]:
%%writefile /content/frontend/app.py
import sys
sys.path.insert(0, '/content')

import streamlit as st
import pandas as pd
import plotly.express as px

from backend.data_loader import (load_data, filter_data, get_locations,
                                  get_pollutant_columns, get_summary_stats, get_aqi_category)
from backend.forecast import get_forecast
from ai.agent import ask_agent, get_quick_insight

st.set_page_config(page_title='Delhi AQI Intelligence', page_icon='🌫️',
                   layout='wide', initial_sidebar_state='expanded')

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Syne:wght@700;800&family=DM+Sans:wght@300;400;500&display=swap');
html,body,[class*="css"]{font-family:'DM Sans',sans-serif;}
h1,h2,h3{font-family:'Syne',sans-serif;}
.stApp{background:#0d1117;color:#e6edf3;}
.card{background:linear-gradient(135deg,#161b22,#1c2128);border:1px solid #30363d;
      border-radius:12px;padding:20px;text-align:center;}
.clabel{font-size:.72rem;color:#8b949e;text-transform:uppercase;letter-spacing:1px;}
.cvalue{font-size:2rem;font-weight:800;font-family:'Syne',sans-serif;}
.csub{font-size:.75rem;color:#8b949e;margin-top:4px;}
.chat-user{background:#1f6feb;color:white;padding:12px 18px;border-radius:18px 18px 4px 18px;
           margin:8px 0 8px auto;max-width:75%;font-size:.95rem;display:block;}
.chat-ai{background:#161b22;border:1px solid #30363d;color:#e6edf3;padding:14px 18px;
         border-radius:18px 18px 18px 4px;margin:8px auto 8px 0;max-width:85%;
         font-size:.95rem;line-height:1.7;}
.chat-wrap{min-height:300px;max-height:500px;overflow-y:auto;
           padding:16px;border:1px solid #21262d;border-radius:12px;
           background:#0d1117;margin-bottom:16px;}
.suggestion-btn{background:#161b22 !important;border:1px solid #30363d !important;
                color:#8b949e !important;border-radius:20px !important;
                font-size:.8rem !important;padding:4px 14px !important;}
div[data-testid="stSidebar"]{background:#0d1117;border-right:1px solid #21262d;}
</style>""", unsafe_allow_html=True)

@st.cache_data
def cached_load():
    return load_data()

st.markdown('# 🌫️ Delhi AQI Intelligence')
st.markdown('##### AI-powered air quality analytics · 2025–2026')
st.divider()

try:
    df_raw = cached_load()
except Exception as e:
    st.error(f'❌ Error loading data: {e}')
    st.stop()

# ── Sidebar ───────────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown('### ⚙️ Filters')
    locs     = get_locations(df_raw)
    sel_locs = st.multiselect('📍 Locations', locs, default=locs[:4] if len(locs)>=4 else locs)
    min_d, max_d = df_raw['datetime'].min().date(), df_raw['datetime'].max().date()
    dr       = st.date_input('📅 Date Range', value=(min_d, max_d), min_value=min_d, max_value=max_d)
    poll_cols= get_pollutant_columns(df_raw)
    sel_poll = st.selectbox('🧪 Metric', poll_cols)
    fc_days  = st.slider('🔮 Forecast Days', 7, 60, 30, 7)
    st.markdown('<small style="color:#8b949e">Streamlit · Prophet · Claude AI</small>',
                unsafe_allow_html=True)

s_d, e_d = (dr[0], dr[1]) if len(dr)==2 else (min_d, max_d)
df = filter_data(df_raw, locations=sel_locs or None, start_date=s_d, end_date=e_d)

if df.empty:
    st.warning('⚠️ No data for selected filters.')
    st.stop()

# ── KPI cards ─────────────────────────────────────────────────────────────────
stats = get_summary_stats(df)
aq    = stats.get('aqi', {})
cur   = aq.get('latest', aq.get('mean', 0)) or 0
cat, aqi_color = get_aqi_category(cur)

c1,c2,c3,c4,c5 = st.columns(5)
for col_obj, label, val, sub, color in [
    (c1,'Latest AQI',  f'{cur:.0f}',                          cat,     aqi_color),
    (c2,'Average AQI', f"{aq.get('mean',0):.0f}",             'Mean',  '#4FC3F7'),
    (c3,'Peak AQI',    f"{aq.get('max',0):.0f}",              'Max',   '#FF7043'),
    (c4,'Records',     f"{stats['_meta']['total_records']:,}", 'Points','#81C784'),
    (c5,'Stations',    str(len(sel_locs or locs)),             'Active','#CE93D8'),
]:
    with col_obj:
        st.markdown(f'<div class="card"><div class="clabel">{label}</div>'
                    f'<div class="cvalue" style="color:{color}">{val}</div>'
                    f'<div class="csub">{sub}</div></div>', unsafe_allow_html=True)

st.markdown('<br>', unsafe_allow_html=True)

tab1, tab2, tab3, tab4 = st.tabs(['📊 Dashboard','🗺️ Heatmap','🔮 Forecast','🤖 AI Chat'])

# ── Tab 1 ─────────────────────────────────────────────────────────────────────
with tab1:
    st.markdown('#### 📈 Trend Over Time')
    fig = px.line(df, x='datetime', y=sel_poll, color='location', template='plotly_dark',
                  labels={'datetime':'Date', sel_poll:sel_poll.upper(), 'location':'Station'})
    fig.update_layout(margin=dict(l=0,r=0,t=30,b=0), hovermode='x unified',
                      legend=dict(orientation='h',y=-0.2))
    st.plotly_chart(fig, use_container_width=True)

    col_a, col_b = st.columns(2)
    with col_a:
        st.markdown('#### 🥧 AQI Category Split')
        if 'aqi' in df.columns and df['aqi'].notna().sum() > 0:
            bins   = [0,50,100,200,300,400,9999]
            labels = ['Good','Satisfactory','Moderate','Poor','Very Poor','Severe']
            colors = ['#00e400','#ffff00','#ff7e00','#ff0000','#8f3f97','#7e0023']
            df2 = df.copy()
            df2['aqi_cat'] = pd.cut(df2['aqi'], bins=bins, labels=labels)
            dist = df2['aqi_cat'].value_counts().reindex(labels).fillna(0)
            fig2 = px.pie(values=dist.values, names=dist.index, hole=0.45,
                          color=dist.index, color_discrete_map=dict(zip(labels,colors)),
                          template='plotly_dark')
            fig2.update_layout(margin=dict(l=0,r=0,t=10,b=0))
            st.plotly_chart(fig2, use_container_width=True)
    with col_b:
        st.markdown('#### 📦 AQI by Station')
        if 'aqi' in df.columns and df['aqi'].notna().sum() > 0:
            fig3 = px.box(df, x='location', y='aqi', color='location', template='plotly_dark')
            fig3.update_layout(showlegend=False, margin=dict(l=0,r=0,t=10,b=0))
            st.plotly_chart(fig3, use_container_width=True)

    st.markdown('#### 🔬 Pollutant Correlation')
    num_cols = [c for c in poll_cols if df[c].notna().sum() > 10]
    if len(num_cols) >= 2:
        fig4 = px.imshow(df[num_cols].corr(), text_auto='.2f', template='plotly_dark',
                         color_continuous_scale='RdBu_r', zmin=-1, zmax=1)
        fig4.update_layout(margin=dict(l=0,r=0,t=30,b=0))
        st.plotly_chart(fig4, use_container_width=True)

# ── Tab 2 ─────────────────────────────────────────────────────────────────────
with tab2:
    st.markdown('#### 🗺️ Monthly AQI Heatmap by Station')
    if 'aqi' in df.columns:
        df3 = df.copy()
        df3['month'] = df3['datetime'].dt.to_period('M').astype(str)
        pivot = df3.groupby(['location','month'])['aqi'].mean().reset_index()
        pivot = pivot.pivot(index='location', columns='month', values='aqi')
        if not pivot.empty:
            fig5 = px.imshow(pivot, color_continuous_scale='YlOrRd', template='plotly_dark',
                             aspect='auto', labels=dict(x='Month',y='Station',color='AQI'))
            fig5.update_layout(margin=dict(l=0,r=0,t=30,b=0))
            st.plotly_chart(fig5, use_container_width=True)

# ── Tab 3 ─────────────────────────────────────────────────────────────────────
with tab3:
    st.markdown('#### 🔮 AQI Forecast')
    fc_loc = st.selectbox('Station to forecast',
                          ['All (Combined)'] + (sel_locs or locs), key='fcloc')
    if st.button('▶ Run Forecast', type='primary'):
        with st.spinner('Training model...'):
            res = get_forecast(df,
                               location=None if fc_loc=='All (Combined)' else fc_loc,
                               target_col='aqi' if 'aqi' in df.columns else poll_cols[0],
                               days=fc_days)
        if 'error' in res:
            st.error(res['error'])
        else:
            f1,f2,f3 = st.columns(3)
            f1.metric('Avg Forecast AQI', res['avg_forecast'])
            f2.metric('Peak Forecast AQI', res['max_forecast'])
            f3.metric('Best Forecast AQI', res['min_forecast'])
            st.plotly_chart(res['fig'], use_container_width=True)
            fcat, fcol = get_aqi_category(res['avg_forecast'])
            st.markdown(f"Forecast: Expected avg AQI **{res['avg_forecast']}** → "
                        f"<span style='color:{fcol};font-weight:700'>{fcat}</span>",
                        unsafe_allow_html=True)
    else:
        st.info('Click **Run Forecast** to generate prediction.')

# ── Tab 4: AI Chat (ChatGPT style) ────────────────────────────────────────────
with tab4:

    # Session state
    if 'history'  not in st.session_state: st.session_state.history  = []
    if 'raw_hist' not in st.session_state: st.session_state.raw_hist = []
    if 'pending'  not in st.session_state: st.session_state.pending  = None

    # ── Process pending question FIRST (before rendering) ──
    if st.session_state.pending:
        question = st.session_state.pending
        st.session_state.pending = None
        st.session_state.history.append({'role':'user','content': question})
        with st.spinner('🤖 Thinking...'):
            reply = ask_agent(question, df, chat_history=st.session_state.raw_hist)
        st.session_state.history.append({'role':'assistant','content': reply})
        st.session_state.raw_hist += [
            {'role':'user',      'content': question},
            {'role':'assistant', 'content': reply},
        ]

    # ── 1. Chat history (top) ──
    if not st.session_state.history:
        st.markdown("""
        <div style="text-align:center;padding:60px 20px;color:#8b949e;">
            <div style="font-size:2.5rem;">🤖</div>
            <div style="font-size:1.1rem;margin-top:12px;font-family:'Syne',sans-serif;">
                Ask me anything about Delhi's air quality
            </div>
            <div style="font-size:.85rem;margin-top:8px;">
                Powered by Claude AI · Answering from your real data
            </div>
        </div>""", unsafe_allow_html=True)
    else:
        for m in st.session_state.history:
            if m['role'] == 'user':
                st.markdown(f'<div class="chat-user">🧑 {m["content"]}</div>',
                            unsafe_allow_html=True)
            else:
                st.markdown(f'<div class="chat-ai">🤖 {m["content"]}</div>',
                            unsafe_allow_html=True)

        if st.button('🗑️ Clear Chat', key='clear'):
            st.session_state.history  = []
            st.session_state.raw_hist = []
            st.rerun()

    st.markdown('<br>', unsafe_allow_html=True)

    # ── 2. Suggestion chips (middle) ──
    st.markdown('<small style="color:#8b949e">💡 Quick questions:</small>',
                unsafe_allow_html=True)
    examples = [
        'Which location had the worst AQI?',
        'Is air quality improving over time?',
        'What does NO2 trend look like?',
        'Is it safe to go outside today?',
        'Which month had the highest pollution?',
        'Compare all stations.',
    ]
    c1,c2,c3 = st.columns(3)
    for i, q in enumerate(examples):
        col = [c1,c2,c3][i%3]
        if col.button(q, key=f'eq{i}', use_container_width=True):
            st.session_state.pending = q
            st.rerun()

    st.markdown('<br>', unsafe_allow_html=True)

    # ── 3. Input box (bottom) ──
    with st.container():
        col_input, col_send = st.columns([6,1])
        user_in = col_input.text_input('',
                                        placeholder='Message the AI analyst...',
                                        label_visibility='collapsed',
                                        key='chat_input')
        send = col_send.button('Send', type='primary', use_container_width=True)

    if send and user_in.strip():
        st.session_state.pending = user_in.strip()
        st.rerun()

Overwriting /content/frontend/app.py


## ✅ CELL 9 — Launch the App

In [ ]:
# Check all files exist
import os
files = [
    '/content/ai/agent.py',
    '/content/ai/__init__.py',
    '/content/backend/data_loader.py',
    '/content/backend/forecast.py',
    '/content/backend/__init__.py',
    '/content/frontend/app.py',
    '/content/data/delhi_aqi.csv',
]
for f in files:
    status = '✅' if os.path.exists(f) else '❌ MISSING'
    print(f'{status} {f}')

✅ /content/ai/agent.py
✅ /content/ai/__init__.py
✅ /content/backend/data_loader.py
✅ /content/backend/forecast.py
✅ /content/backend/__init__.py
✅ /content/frontend/app.py
✅ /content/data/delhi_aqi.csv


In [ ]:
import subprocess, sys, time, os, threading, shutil

# Kill old processes
subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(2)

# Set your API key
os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-your-key-here'  # ← paste api key

# Start Streamlit
st_path = shutil.which('streamlit') or f"{sys.prefix}/bin/streamlit"
subprocess.Popen(
    [st_path, 'run', '/content/frontend/app.py',
     '--server.port=8501', '--server.headless=true'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    env=os.environ.copy()
)
time.sleep(6)
print('✅ Streamlit started!')

# Fresh Cloudflare tunnel
subprocess.run(['wget', '-q',
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    '-O', '/usr/local/bin/cloudflared'], capture_output=True)
subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'])

tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
                           stdout=subprocess.PIPE, stderr=subprocess.PIPE)

def read(pipe):
    for line in pipe:
        text = line.decode().strip()
        if 'trycloudflare.com' in text:
            url = [w for w in text.split() if 'trycloudflare.com' in w][0]
            if not url.startswith('http'): url = 'https://' + url
            print(f'\n🚀 New URL: {url}')

threading.Thread(target=read, args=(tunnel.stderr,), daemon=True).start()
time.sleep(10)

✅ Streamlit started!

🚀 New URL: https://trycloudflare.com...

🚀 New URL: https://prisoners-ceramic-congressional-happens.trycloudflare.com
